# BrainDecoding — Pallier2025 云端全任务一键执行流水线（按任务分单元格）

本 Notebook 专为云端（Google Colab GPU）执行 `BrainDecoding/runs/pallier2025` 全部实验补齐与重算任务而设计，**每个代码单元格独立对应一项具体任务**：

- **单元格 1（环境准备）**：挂载 Google Drive、拉取最新代码、初始化防崩虚拟环境、建立 `/content/studies_links` 软链接桥并秒级解包 `features_meg.tar`（1.0s 动态切窗缓存）；
- **单元格 2（任务 1 — 既有 1.0s 模型报表同步）**：0 秒刷新并校验 `Word-Only`、`+ Causal CMS (r=0)`、`Local-Window Revision (r=2)`、`Prefix-Revision (Full)` 的全量测试集 ($N=18,450$) 多种子报告与跨模型统计检验；
- **单元格 3（任务 2A — `Full-Context (Joint, Unfrozen CNN)` 1.0s 直接覆盖重训与双口径评估）**：直接覆盖 `stage1_lr4e-4_bs256_stage2_lr1e-4_bs128/context` 下旧版 0.5s 产物，重训 1.0s 窗口 3 种子 (`seed=0,1,2`) 并导出全量集 ($N=18,450$) 与 $K_2$ 同交集子集 ($N=14,590$) 报告；
- **单元格 4（任务 2B — `+ Freeze CNN` 1.0s 直接覆盖重训与双口径评估）**：直接覆盖 `stage1_freeze_bs256_stage2_lr1e-4_bs128/context` 下旧版 0.5s 产物，重训 1.0s 窗口 3 种子 (`seed=0,1,2`) 并导出全量集 ($N=18,450$) 与 $K_2$ 同交集子集 ($N=14,590$) 报告；
- **单元格 5（任务 3 — `Word-Only` 与 `+ Causal CMS (r=0)` 同交集子集 $N=14,590$ 秒级复评）**：直接复用云盘已有 `subspace_encoded_eval_test.pt` 特征缓存，在严格对齐 $K_2$ 的 $N=14,590$ 词事件（$7,440$ 唯一目标词查询）子集上秒级完成多种子评测并独立落盘 `common_subset_*` 报告；
- **单元格 6（任务 4 — `Prefix-Revision w/o L_delta-null` 消融补齐 `seed-001, 002` 与 3 种子评估）**：在 `lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window/prefix_revision` 下补训缺失的 `seed=1,2`（`--lambda-delta-null 0.0 --beta-delta-cm 0.0`），并完成 3 种子联合测试集评估；
- **单元格 7（任务 5 — 全词表开放句段流式解码 `sequence_decoding` $|V|=2,020, n=1,940$ 补齐与 3 种子汇总）**：在 `2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window/sequence_decoding` 下零开销复用 `seed-000` 已完成解码 JSON 及 `seed-001, 002` 已有 `.pt`/校准文件，补齐 `seed-001, 002` 束搜索并生成 3 种子汇总报告；
- **单元格 8（任务 6 — 云盘全任务产物完整性核验与双口径总表打印）**：一键扫描核对五大实验目录的全部 JSON/TXT 汇总表并打印全量集 ($N=18,450$) 与同交集子集 ($N=14,590$) 对比看板。

## 单元格 1：云端环境初始化、Google Drive 挂载、软链接桥与 1.0s 信号缓存秒级解包

In [ ]:
# 单元格 1：挂载 Google Drive、同步最新代码、初始化虚拟环境并解包 features_meg.tar
!nvidia-smi

import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 全局路径与仓库配置 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'
DATASET = 'pallier2025'
MODALITY = 'meg'

# 五大核心实验目录标签
TAG_WORD_ONLY = 'lr4e-4_bs256'
TAG_JOINT_UNFROZEN_CTX = 'stage1_lr4e-4_bs256_stage2_lr1e-4_bs128'
TAG_FREEZE_CNN_CTX = 'stage1_freeze_bs256_stage2_lr1e-4_bs128'
TAG_SIAMESE_HISTORY = 'lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'
TAG_PREFIX_FULL = '2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

def run_cmd(cmd: str, env_overrides: dict | None = None) -> None:
    """在 /content/code 下执行命令并实时流式打印输出，遇到非零退出码立即抛错。"""
    print(f'\n>>> {cmd}', flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if env_overrides:
        for k, v in env_overrides.items():
            if v is None:
                env.pop(k, None)
            else:
                env[k] = str(v)
    proc = subprocess.Popen(
        cmd,
        shell=True,
        cwd=CODE_DIR if Path(CODE_DIR).is_dir() else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'命令执行失败 (exit={ret}): {cmd}')

# 1. 克隆或同步最新代码
if not Path(f'{CODE_DIR}/.git').is_dir():
    run_cmd(f'git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}')
else:
    run_cmd(f'git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only')

# 2. 初始化防崩虚拟环境与软链接桥
run_cmd(
    f'python3 -u {CODE_DIR}/scripts/colab_setup.py '
    f'--venv-dir {VENV_DIR} '
    f'--project-root {CODE_DIR} '
    f'--drive-data-dir {DRIVE_DATA_DIR} '
    f'--links-dir {LINKS_DIR} '
    f'--local-cache-dir {LOCAL_CACHE_DIR}'
)

# 3. 秒级解包 features_meg.tar 到本地高速盘 /content/local_cache
run_cmd(
    f'{PYTHON} -u scripts/cache_archive.py unpack '
    f'--archive-dir {DRIVE_ARCHIVE_DIR} '
    f'--cache-dir {LOCAL_CACHE_DIR} '
    f'--modality {MODALITY} '
    f'--skip-existing'
)
print('\n[OK] 云端环境、软链接桥与 1.0s MEG 缓存准备完毕！')

## 单元格 2（任务 1）：既有 1.0s 模型全量集 ($N=18,450$) 多种子报告同步与四模型配对统计核验

In [ ]:
# 单元格 2（任务 1）：0 秒刷新既有 1.0s 模型的全量测试集 (N=18,450) 多种子汇总与跨模型配对检验
for exp_tag, stage_name in [
    (TAG_WORD_ONLY, 'word'),
    (TAG_SIAMESE_HISTORY, 'context'),
    (TAG_SIAMESE_HISTORY, 'revision'),
    (TAG_PREFIX_FULL, 'prefix_revision'),
]:
    target_stage_dir = f'{DRIVE_RESULTS_DIR}/{DATASET}/{exp_tag}/{stage_name}'
    run_cmd(
        f'{PYTHON} -u -m braindecoding report '
        f'--input {target_stage_dir}',
        env_overrides={'BRAINDECODING_EXP_TAG': exp_tag, 'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY},
    )

## 单元格 3（任务 2A）：`Full-Context (Joint, Unfrozen CNN)` (`stage1_lr4e-4_bs256_stage2_lr1e-4_bs128`) 1.0s 窗口直接覆盖重训与双口径评估

- **配置**：`configs/training/pallier2025_context_transformer.yaml`（`window_seconds: 1.0`、`unfreeze_brain_encoder_from_epoch: 1`、学习率 `1e-4`、批大小 `128`，从 `lr4e-4_bs256` 热启动 Stage 1 权重）
- **落盘策略**：直接覆盖 (`--overwrite`) `stage1_lr4e-4_bs256_stage2_lr1e-4_bs128/context` 下旧版 0.5s 产物，无需备份；
- **双口径评估**：先完成全量集 ($N=18,450$, $9,225$ queries) 3 种子评估与 `.pt` 特征缓存导出，再 0 秒导出对齐 $K_2$ 视界的同交集子集 ($N=14,590$, $7,440$ queries) 报告。

In [ ]:
# 单元格 3（任务 2A）：Full-Context (Joint, Unfrozen CNN) 1.0s 直接覆盖重训 + 全量集(N=18,450)与同交集子集(N=14,590)评估
import yaml

BASE_CTX_YAML = Path(CODE_DIR) / 'configs/training/pallier2025_context_transformer.yaml'
UNFROZEN_CTX_YAML = Path('/tmp/pallier2025_unfrozen_cnn_context_1.0s.yaml')

cfg_2a = yaml.safe_load(BASE_CTX_YAML.read_text(encoding='utf-8'))
cfg_2a['extends'] = str(Path(CODE_DIR) / 'configs/word_decoding/pallier2025/all_subjects/main_context_warmstart.yaml')
cfg_2a.setdefault('workflow', {})['window_seconds'] = 1.0
cfg_2a['workflow']['exp_tag'] = TAG_JOINT_UNFROZEN_CTX
cfg_2a['workflow']['word_exp_tag'] = TAG_WORD_ONLY
cfg_2a.setdefault('training', {})['freeze_brain_encoder_epochs'] = 1
UNFROZEN_CTX_YAML.write_text(yaml.safe_dump(cfg_2a, allow_unicode=True, sort_keys=False), encoding='utf-8')

CTRL_SUBSPACE = 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'

env_2a = {
    'BRAINDECODING_EXP_TAG': TAG_JOINT_UNFROZEN_CTX,
    'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY,
}

# 1. 若 3 个种子的 1.0s 权重 (freeze_brain_encoder_epochs=1) 尚未全部就绪，则直接覆盖 (--overwrite) 重训
_ctx_2a_dir = Path(DRIVE_RESULTS_DIR) / DATASET / TAG_JOINT_UNFROZEN_CTX / 'context' / 'all-n10'
def _is_unfrozen_1s_seed_ready(seed_dir: Path) -> bool:
    if not (seed_dir / 'best_full_vocab.pt').is_file() or not (seed_dir / 'resolved_config.yaml').is_file():
        return False
    rc = yaml.safe_load((seed_dir / 'resolved_config.yaml').read_text(encoding='utf-8')) or {}
    return int((rc.get('training') or {}).get('freeze_brain_encoder_epochs', 0)) == 1

_ready_2a = all(_is_unfrozen_1s_seed_ready(_ctx_2a_dir / f'seed-{s:03d}') for s in (0, 1, 2))
if _ready_2a:
    print(f'[SKIP TRAIN] {TAG_JOINT_UNFROZEN_CTX} (freeze_brain_encoder_epochs=1) 的 seed-000..002 1.0s 权重已全部训练完成，直接进入评估！')
else:
    run_cmd(
        f'{PYTHON} -u -m braindecoding train '
        f'--config {UNFROZEN_CTX_YAML} '
        f'--exp-tag {TAG_JOINT_UNFROZEN_CTX} '
        f'--word-exp-tag {TAG_WORD_ONLY} '
        f'--window-seconds 1.0 '
        f'--seed 0,1,2 '
        f'--require-cache '
        f'--overwrite '
        f'--subject all',
        env_overrides=env_2a,
    )

# 2. 全量测试集 (N=18,450, 9,225 queries) 3 种子评估 + 导出 subspace_encoded_eval_test.pt
run_cmd(
    f'{PYTHON} -u -m braindecoding eval-run '
    f'--config {UNFROZEN_CTX_YAML} '
    f'--control-config {CTRL_SUBSPACE} '
    f'--exp-tag {TAG_JOINT_UNFROZEN_CTX} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--stage context '
    f'--window-seconds 1.0 '
    f'--seed 0,1,2 '
    f'--split test '
    f'--require-cache '
    f'--subject all',
    env_overrides=env_2a,
)

# 3. 基于刚导出的 .pt 缓存 0 秒评测 K2 同交集子集 (N=14,590, 7,440 queries)
run_cmd(
    f'{PYTHON} -u -m braindecoding eval-run '
    f'--config {UNFROZEN_CTX_YAML} '
    f'--control-config {CTRL_SUBSPACE} '
    f'--exp-tag {TAG_JOINT_UNFROZEN_CTX} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--stage context '
    f'--window-seconds 1.0 '
    f'--common-horizon-delay 2 '
    f'--seed 0,1,2 '
    f'--split test '
    f'--require-cache '
    f'--subject all',
    env_overrides=env_2a,
)

## 单元格 4（任务 2B）：`+ Freeze CNN` (`stage1_freeze_bs256_stage2_lr1e-4_bs128`) 1.0s 窗口直接覆盖重训与双口径评估

- **配置**：基于 `configs/training/pallier2025_context_transformer.yaml`，将 `unfreeze_brain_encoder_from_epoch` 设为 `999`（全程冻结 Stage 1 CNN 编码器，启用零冗余预提取加速），`window_seconds: 1.0`；
- **落盘策略**：直接覆盖 (`--overwrite`) `stage1_freeze_bs256_stage2_lr1e-4_bs128/context` 下旧版 0.5s 产物，无需备份；
- **双口径评估**：先完成全量集 ($N=18,450$, $9,225$ queries) 3 种子评估与 `.pt` 特征缓存导出，再 0 秒导出对齐 $K_2$ 视界的同交集子集 ($N=14,590$, $7,440$ queries) 报告。

In [ ]:
# 单元格 4（任务 2B）：+ Freeze CNN (stage1_freeze_bs256_stage2_lr1e-4_bs128) 1.0s 直接覆盖重训 + 双口径评估
import shutil
import yaml

BASE_CTX_YAML = Path(CODE_DIR) / 'configs/training/pallier2025_context_transformer.yaml'
FREEZE_CTX_YAML = Path('/tmp/pallier2025_freeze_cnn_context_1.0s.yaml')

cfg_data = yaml.safe_load(BASE_CTX_YAML.read_text(encoding='utf-8'))
cfg_data['extends'] = str(Path(CODE_DIR) / 'configs/word_decoding/pallier2025/all_subjects/main_context_warmstart.yaml')
cfg_data.setdefault('workflow', {})['window_seconds'] = 1.0
cfg_data['workflow']['exp_tag'] = TAG_FREEZE_CNN_CTX
cfg_data['workflow']['word_exp_tag'] = TAG_WORD_ONLY
cfg_data.setdefault('training', {})['freeze_brain_encoder_epochs'] = 30
FREEZE_CTX_YAML.write_text(yaml.safe_dump(cfg_data, allow_unicode=True, sort_keys=False), encoding='utf-8')

CTRL_SUBSPACE = 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'
env_2b = {
    'BRAINDECODING_EXP_TAG': TAG_FREEZE_CNN_CTX,
    'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY,
}

_ctx_2a_dir = Path(DRIVE_RESULTS_DIR) / DATASET / TAG_JOINT_UNFROZEN_CTX / 'context' / 'all-n10'
_ctx_2b_dir = Path(DRIVE_RESULTS_DIR) / DATASET / TAG_FREEZE_CNN_CTX / 'context' / 'all-n10'

def _is_frozen_1s_seed_ready(seed_dir: Path) -> bool:
    if not (seed_dir / 'best_full_vocab.pt').is_file() or not (seed_dir / 'resolved_config.yaml').is_file():
        return False
    rc = yaml.safe_load((seed_dir / 'resolved_config.yaml').read_text(encoding='utf-8')) or {}
    return int((rc.get('training') or {}).get('freeze_brain_encoder_epochs', 0)) >= 30

# 若单元格 3 刚才用默认 pallier2025_context_transformer.yaml (freeze_brain_encoder_epochs=30) 跑完了 3 种子，直接无损归位至 TAG_FREEZE_CNN_CTX
if not all(_is_frozen_1s_seed_ready(_ctx_2b_dir / f'seed-{s:03d}') for s in (0, 1, 2)):
    if all(_is_frozen_1s_seed_ready(_ctx_2a_dir / f'seed-{s:03d}') for s in (0, 1, 2)):
        print(f'[MIGRATE] 检测到 {TAG_JOINT_UNFROZEN_CTX} 下已存在 freeze_brain_encoder_epochs=30 的 1.0s 完整结果，正在直接归位至 {TAG_FREEZE_CNN_CTX} ...')
        if _ctx_2b_dir.exists():
            shutil.rmtree(_ctx_2b_dir)
        shutil.copytree(_ctx_2a_dir, _ctx_2b_dir)
        print(f'[OK] 已归位至 {_ctx_2b_dir}，无需重复训练！')

# 1. 若 3 个种子的 1.0s 冻结权重尚未全部就绪，则直接覆盖 (--overwrite) 重训，全程冻结 CNN 并启用预提取加速
_ready_2b = all(_is_frozen_1s_seed_ready(_ctx_2b_dir / f'seed-{s:03d}') for s in (0, 1, 2))
if _ready_2b:
    print(f'[SKIP TRAIN] {TAG_FREEZE_CNN_CTX} (freeze_brain_encoder_epochs=30) 的 seed-000..002 1.0s 权重已全部就绪！')
else:
    run_cmd(
        f'{PYTHON} -u -m braindecoding train '
        f'--config {FREEZE_CTX_YAML} '
        f'--exp-tag {TAG_FREEZE_CNN_CTX} '
        f'--word-exp-tag {TAG_WORD_ONLY} '
        f'--window-seconds 1.0 '
        f'--preextract-frozen-cnn '
        f'--seed 0,1,2 '
        f'--require-cache '
        f'--overwrite '
        f'--subject all',
        env_overrides=env_2b,
    )

# 2. 全量测试集 (N=18,450, 9,225 queries) 3 种子评估（若已由归位携带有 multi_seed_evaluation_test.json 则仅刷新报表，否则跑完整评估）
if (_ctx_2b_dir / 'multi_seed_evaluation_test.json').is_file() and all((_ctx_2b_dir / f'seed-{s:03d}' / 'exploratory' / 'subspace_encoded_eval_test.pt').is_file() for s in (0, 1, 2)):
    run_cmd(
        f'{PYTHON} -u -m braindecoding report --input {_ctx_2b_dir.parent}',
        env_overrides=env_2b,
    )
else:
    run_cmd(
        f'{PYTHON} -u -m braindecoding eval-run '
        f'--config {FREEZE_CTX_YAML} '
        f'--control-config {CTRL_SUBSPACE} '
        f'--exp-tag {TAG_FREEZE_CNN_CTX} '
        f'--word-exp-tag {TAG_WORD_ONLY} '
        f'--stage context '
        f'--window-seconds 1.0 '
        f'--seed 0,1,2 '
        f'--split test '
        f'--require-cache '
        f'--subject all',
        env_overrides=env_2b,
    )

# 3. 基于 .pt 缓存 0 秒评测 K2 同交集子集 (N=14,590, 7,440 queries)
run_cmd(
    f'{PYTHON} -u -m braindecoding eval-run '
    f'--config {FREEZE_CTX_YAML} '
    f'--control-config {CTRL_SUBSPACE} '
    f'--exp-tag {TAG_FREEZE_CNN_CTX} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--stage context '
    f'--window-seconds 1.0 '
    f'--common-horizon-delay 2 '
    f'--seed 0,1,2 '
    f'--split test '
    f'--require-cache '
    f'--subject all',
    env_overrides=env_2b,
)

## 单元格 5（任务 3）：`Word-Only` 与 `+ Causal CMS (r=0)` 同交集子集 ($N=14,590, 7,440\text{ queries}$) 秒级复评

- 直接读取云盘已有的 `subspace_encoded_eval_test.pt`，通过 `--common-horizon-delay 2` 将评测事件严格过滤为与 $K_2$ 视界完全同源的 $N=14,590$ 词事件（$7,440$ 唯一目标词查询）；
- **不覆盖**原有全量集 `evaluation_test.json` / `multi_seed_evaluation_test.json`，而是独立输出 `evaluation_common_subset_test.json`、`multi_seed_evaluation_common_subset_test.json` 与 `multi_seed_evaluation_common_subset_report_test.txt`。

In [ ]:
# 单元格 5（任务 3）：Word-Only 与 + Causal CMS (r=0) 在 N=14,590 (7,440 queries) 同交集子集上秒级复评
CTRL_SUBSPACE = 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'

# 1. Word-Only (lr4e-4_bs256/word) 同交集子集 N=14,590 评估
run_cmd(
    f'{PYTHON} -u -m braindecoding eval-run '
    f'--config configs/training/pallier2025_word_encoder.yaml '
    f'--control-config {CTRL_SUBSPACE} '
    f'--exp-tag {TAG_WORD_ONLY} '
    f'--stage word '
    f'--window-seconds 1.0 '
    f'--common-horizon-delay 2 '
    f'--seed 0,1,2 '
    f'--split test '
    f'--require-cache '
    f'--subject all',
    env_overrides={'BRAINDECODING_EXP_TAG': TAG_WORD_ONLY, 'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY},
)

# 2. + Causal CMS (r=0) (lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window/context) 同交集子集 N=14,590 评估
run_cmd(
    f'{PYTHON} -u -m braindecoding eval-run '
    f'--config configs/training/siamese_debiased/pallier2025_siamese_debiased_context.yaml '
    f'--control-config {CTRL_SUBSPACE} '
    f'--exp-tag {TAG_SIAMESE_HISTORY} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--stage context '
    f'--window-seconds 1.0 '
    f'--common-horizon-delay 2 '
    f'--seed 0,1,2 '
    f'--split test '
    f'--require-cache '
    f'--subject all',
    env_overrides={'BRAINDECODING_EXP_TAG': TAG_SIAMESE_HISTORY, 'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY},
)

## 单元格 6（任务 4）：`Prefix-Revision w/o L_delta-null` 消融实验补齐 (`seed-001, seed-002`) 与 3 种子评估

- **目录**：`lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window/prefix_revision`（其中 `seed-000` 已完成，缺失 `seed-001, seed-002`）；
- **消融超参**：与 `seed-000` 严格一致，固定 `--eta-0 0.50 --eta-delta 0.50 --lambda-null 10.0 --lambda-delta-null 0.0 --beta-delta-cm 0.0 --shifts-per-step 6`；
- **评估**：训练完成后对 `seed=0,1,2` 运行完整 3 种子测试集评估，生成 `multi_seed_evaluation_test.json` 与 `multi_seed_evaluation_report_test.txt`。

In [ ]:
# 单元格 6（任务 4）：补训 Prefix-Revision w/o L_delta-null 的 seed-001, seed-002 并运行 3 种子联合评估
CFG_PREFIX_REV = 'configs/training/siamese_debiased/pallier2025_siamese_debiased_prefix_revision.yaml'
CTRL_SUBSPACE = 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'

env_task4 = {
    'BRAINDECODING_EXP_TAG': TAG_SIAMESE_HISTORY,
    'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY,
}

# 1. 仅补训缺失的 seed=1,2（保留已完成的 seed-000）
run_cmd(
    f'{PYTHON} -u -m braindecoding train '
    f'--config {CFG_PREFIX_REV} '
    f'--exp-tag {TAG_SIAMESE_HISTORY} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--window-seconds 1.0 '
    f'--eta-0 0.50 '
    f'--eta-delta 0.50 '
    f'--lambda-null 10.0 '
    f'--lambda-delta-null 0.0 '
    f'--beta-delta-cm 0.0 '
    f'--shifts-per-step 6 '
    f'--seed 1,2 '
    f'--require-cache '
    f'--resume auto '
    f'--subject all',
    env_overrides=env_task4,
)

# 2. 汇总评估全部 3 个随机种子 (seed=0,1,2)
run_cmd(
    f'{PYTHON} -u -m braindecoding eval-run '
    f'--config {CFG_PREFIX_REV} '
    f'--control-config {CTRL_SUBSPACE} '
    f'--exp-tag {TAG_SIAMESE_HISTORY} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--stage prefix_revision '
    f'--window-seconds 1.0 '
    f'--seed 0,1,2 '
    f'--split test '
    f'--require-cache '
    f'--subject all',
    env_overrides=env_task4,
)

## 单元格 7（任务 5）：全词表开放句段流式解码 (`sequence_decoding`, $|V|=2,020, n=1,940$) 3 种子补齐与汇总

- **目标目录**：`2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window/sequence_decoding`
- **智能缓存复用**：
  - `seed-000`：直接复用已完成的 `.pt`、校准文件与 `sequence_decoding_test.json`（0 秒跳过束搜索，直接参与评估汇总）；
  - `seed-001`：直接复用已完成的 `neural_vocabulary_scores_{val,test}.pt` 与 `state_hyperparameter_calibration.json`，仅执行束搜索解码；
  - `seed-002`：自动跨子目录只读复用 `simple_sequence_decoding/seed-002` 已导出的 `.pt` 与 `state_hyperparameter_calibration.json`，免去神经前向与验证集网格校准，直接执行束搜索解码；
- **产物**：生成 3 种子完整的 `multi_seed_sequence_decoding_test.json` 与 `multi_seed_sequence_decoding_report_test.txt`（覆盖 $|V|=2,020$ 全句段 $n=1,940$、$|V|=250/100$ 核心词表及 $k\in\{1,2,5,10\}$ 跨被试神经聚合）。

In [ ]:
# 单元格 7（任务 5）：全词表开放句段流式束搜索解码 (sequence_decoding, |V|=2,020, n=1,940) 3 种子补齐与汇总
env_task5 = {
    'BRAINDECODING_EXP_TAG': TAG_PREFIX_FULL,
    'BRAINDECODING_WORD_EXP_TAG': TAG_WORD_ONLY,
    'BRAINDECODING_SEQUENCE_SUBDIR': 'sequence_decoding',
}

run_cmd(
    f'{PYTHON} -u -m braindecoding decode-sentences '
    f'--config configs/decoding/sequence_decoding_protocol.yaml '
    f'--dataset {DATASET} '
    f'--split test '
    f'--seeds 0,1,2 '
    f'--sequence-subdir sequence_decoding '
    f'--exp-tag {TAG_PREFIX_FULL} '
    f'--word-exp-tag {TAG_WORD_ONLY} '
    f'--context-exp-tag none '
    f'--revision-exp-tag {TAG_PREFIX_FULL} '
    f'--revision-stage prefix_revision '
    f'--window-seconds 1.0 '
    f'--min-sentence-length 0',
    env_overrides=env_task5,
)

## 单元格 8（任务 6）：全任务产物完整性核验与双口径 ($N=18,450$ / $N=14,590$) 汇总看板打印

In [ ]:
# 单元格 8（任务 6）：核验云盘五大实验目录的全部产物并打印双口径对比汇总表
import json

runs_root = Path(DRIVE_RESULTS_DIR) / DATASET

def _fmt_mean_sd(block: dict | None, key: str = 'macro_top10') -> str:
    if not isinstance(block, dict):
        return '—'
    sub = block.get(key) if isinstance(block.get(key), dict) else block
    if not isinstance(sub, dict) or 'mean' not in sub:
        return '—'
    m = float(sub['mean']) * 100.0
    s = float(sub.get('std', sub.get('sd', 0.0))) * 100.0
    return f'{m:.2f} ± {s:.2f}%'

print('=' * 100)
print(' [Pallier2025 云端全任务产物核验与核心指标总览 (3 Seeds Mean ± SD)]')
print('=' * 100)

def _resolve_stage_file(base_dir: Path, stage: str, filename: str) -> Path:
    p_n10 = base_dir / stage / 'all-n10' / filename
    if p_n10.is_file():
        return p_n10
    p_all = base_dir / stage / 'all-subjects' / filename
    return p_all if p_all.is_file() else p_n10

check_items = [
    ('Word-Only (Full N=18,450)', _resolve_stage_file(runs_root / TAG_WORD_ONLY, 'word', 'multi_seed_evaluation_test.json')),
    ('Word-Only (Common N=14,590)', _resolve_stage_file(runs_root / TAG_WORD_ONLY, 'word', 'multi_seed_evaluation_common_subset_test.json')),
    ('Full-Context Unfrozen (Full N=18,450)', _resolve_stage_file(runs_root / TAG_JOINT_UNFROZEN_CTX, 'context', 'multi_seed_evaluation_test.json')),
    ('Full-Context Unfrozen (Common N=14,590)', _resolve_stage_file(runs_root / TAG_JOINT_UNFROZEN_CTX, 'context', 'multi_seed_evaluation_common_subset_test.json')),
    ('+ Freeze CNN (Full N=18,450)', _resolve_stage_file(runs_root / TAG_FREEZE_CNN_CTX, 'context', 'multi_seed_evaluation_test.json')),
    ('+ Freeze CNN (Common N=14,590)', _resolve_stage_file(runs_root / TAG_FREEZE_CNN_CTX, 'context', 'multi_seed_evaluation_common_subset_test.json')),
    ('+ Causal CMS r=0 (Full N=18,450)', _resolve_stage_file(runs_root / TAG_SIAMESE_HISTORY, 'context', 'multi_seed_evaluation_test.json')),
    ('+ Causal CMS r=0 (Common N=14,590)', _resolve_stage_file(runs_root / TAG_SIAMESE_HISTORY, 'context', 'multi_seed_evaluation_common_subset_test.json')),
    ('Local-Window Revision r=2 (K0/K1/K2)', _resolve_stage_file(runs_root / TAG_SIAMESE_HISTORY, 'revision', 'multi_seed_evaluation_test.json')),
    ('Prefix-Revision w/o L_delta-null', _resolve_stage_file(runs_root / TAG_SIAMESE_HISTORY, 'prefix_revision', 'multi_seed_evaluation_test.json')),
    ('Prefix-Revision Full (Main Model)', _resolve_stage_file(runs_root / TAG_PREFIX_FULL, 'prefix_revision', 'multi_seed_evaluation_test.json')),
    ('S3 L>=5 Closed-Set (simple_sequence_decoding)', runs_root / TAG_PREFIX_FULL / 'simple_sequence_decoding/multi_seed_sequence_decoding_test.json'),
    ('S3 Open-Segment Full-Vocab (sequence_decoding)', runs_root / TAG_PREFIX_FULL / 'sequence_decoding/multi_seed_sequence_decoding_test.json'),
]

for label, jpath in check_items:
    status = 'OK' if jpath.is_file() else 'MISSING'
    detail = ''
    if jpath.is_file():
        try:
            data = json.loads(jpath.read_text(encoding='utf-8'))
            n_seeds = data.get('num_seeds', len(data.get('seeds', [])))
            detail = f'seeds={n_seeds}'
        except Exception as exc:
            detail = f'read_error={exc}'
    print(f'[{status:7s}] {label:48s} -> {jpath.relative_to(runs_root)} ({detail})')